# Radon Maintainability Rating (SQALE A–E) — Raw Output Extraction (Python)

This notebook analyzes **Python repositories** with **Radon** and captures complete raw tool output for Maintainability Index, Maintainability Rating (A–E), Cyclomatic Complexity, Raw Metrics, and Halstead Metrics.

**Default benchmark repository:** [pallets/flask](https://github.com/pallets/flask)


## Section 1 — Install Dependencies

Install open-source Python packages and verify Radon.


In [ ]:
!pip install -q radon pandas gitpython jupyter

import subprocess, sys
subprocess.run([sys.executable, '-m', 'radon', '--version'], check=False)


## Section 2 — Configuration

Set repository source, workspace, and output directory.


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/pallets/flask.git'

LOCAL_REPO_PATH = '/content/flask'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

STREAM_RAW_OUTPUT = False

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/sqale_rating_benchmark'


## Section 3 — Imports and Utility Functions


In [ ]:
from pathlib import Path

from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "venv", ".venv", "env", "__pycache__", "build", "dist", ".tox", "node_modules", "site-packages",
}
PY = sys.executable


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        if not self.error_log_path.exists():
            self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, file: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "file": file, "error_message": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "file", "error_message"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str, workspace_dir: Path, if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError("IF_CLONE_EXISTS must be 'reuse' or 'reclone'.")
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", file=repo_url)
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise NotADirectoryError(msg)
    try:
        Repo(local_repo_path)
        logger.info("Validated Git repository.")
    except InvalidGitRepositoryError:
        logger.info("Path is not a Git repository; proceeding as source directory.")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool, repo_url: str, local_repo_path: str | Path, workspace_dir: Path,
    if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo_path), logger)


def discover_python_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for path in repo_path.rglob("*.py"):
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def compute_repository_stats(repo_path: Path, python_files: list[Path]) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in python_files)
    directories = {path.parent for path in python_files}
    return {
        "repository_name": repo_path.name,
        "repository_size_bytes": total_size,
        "directory_count": len(directories),
        "python_file_count": len(python_files),
    }


def save_python_inventory(python_files: list[Path], output_csv: Path) -> None:
    pd.DataFrame(
        [{"file_path": str(p), "file_name": p.name, "directory": str(p.parent)} for p in python_files]
    ).to_csv(output_csv, index=False)


def build_radon_command(repo_path: Path, subcommand: str, *, json_output: bool = False, extra_args: list[str] | None = None) -> list[str]:
    command = [PY, "-m", "radon", subcommand, str(repo_path)]
    if subcommand in {"cc", "mi"}:
        command.append("-s")
    if extra_args:
        command.extend(extra_args)
    if json_output:
        command.append("-j")
    return command


def run_radon_command(command: list[str], logger: NotebookLogger, stream_raw: bool = False) -> tuple[str, str, int]:
    if stream_raw:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, encoding="utf-8", errors="replace",
        )
        stdout_lines: list[str] = []
        stderr_lines: list[str] = []
        assert process.stdout is not None
        assert process.stderr is not None
        while True:
            stdout_line = process.stdout.readline()
            stderr_line = process.stderr.readline()
            if stdout_line:
                stdout_lines.append(stdout_line)
                print(stdout_line, end="")
            if stderr_line:
                stderr_lines.append(stderr_line)
                print(stderr_line, end="")
            if not stdout_line and not stderr_line and process.poll() is not None:
                break
        return "".join(stdout_lines), "".join(stderr_lines), process.returncode or 0

    completed = subprocess.run(
        command, capture_output=True, text=True, encoding="utf-8", errors="replace", check=False,
    )
    return completed.stdout, completed.stderr, completed.returncode


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def parse_json_payload(text: str) -> dict[str, Any]:
    if not text.strip():
        return {}
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        return {}
    return payload if isinstance(payload, dict) else {}


def parse_cc_results(payload: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for file_path, blocks in payload.items():
        if not isinstance(blocks, list):
            continue
        for block in blocks:
            rows.append({
                "file": file_path,
                "function": block.get("name", ""),
                "complexity": block.get("complexity", ""),
                "rank": block.get("rank", ""),
                "line": block.get("lineno", ""),
            })
    return pd.DataFrame(rows, columns=["file", "function", "complexity", "rank", "line"])


def parse_mi_results(payload: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for file_path, metrics in payload.items():
        if not isinstance(metrics, dict):
            continue
        rows.append({
            "file": file_path,
            "maintainability_index": metrics.get("mi", ""),
            "rank": metrics.get("rank", ""),
        })
    return pd.DataFrame(rows, columns=["file", "maintainability_index", "rank"])


def parse_raw_results(payload: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for file_path, metrics in payload.items():
        if not isinstance(metrics, dict):
            continue
        rows.append({
            "file": file_path,
            "loc": metrics.get("loc", ""),
            "lloc": metrics.get("lloc", ""),
            "sloc": metrics.get("sloc", ""),
            "comments": metrics.get("comments", ""),
            "multi": metrics.get("multi", ""),
            "blank": metrics.get("blank", ""),
        })
    return pd.DataFrame(rows, columns=["file", "loc", "lloc", "sloc", "comments", "multi", "blank"])


def parse_halstead_results(payload: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for file_path, metrics in payload.items():
        if not isinstance(metrics, dict):
            continue
        total = metrics.get("total", {})
        if not isinstance(total, dict):
            continue
        rows.append({
            "file": file_path,
            "h1": total.get("h1", ""),
            "h2": total.get("h2", ""),
            "N1": total.get("N1", ""),
            "N2": total.get("N2", ""),
            "vocabulary": total.get("vocabulary", ""),
            "length": total.get("length", ""),
            "volume": total.get("volume", ""),
            "difficulty": total.get("difficulty", ""),
            "effort": total.get("effort", ""),
            "bugs": total.get("bugs", ""),
            "time": total.get("time", ""),
        })
    return pd.DataFrame(
        rows,
        columns=["file", "h1", "h2", "N1", "N2", "vocabulary", "length", "volume", "difficulty", "effort", "bugs", "time"],
    )


def mi_to_sqale_rating(mi: float) -> str:
    if mi >= 85:
        return "A"
    if mi >= 70:
        return "B"
    if mi >= 55:
        return "C"
    if mi >= 40:
        return "D"
    return "E"


def compute_maintainability_summary(mi_df: pd.DataFrame) -> pd.DataFrame:
    mi_values = pd.to_numeric(mi_df["maintainability_index"], errors="coerce").dropna()
    avg_mi = round(float(mi_values.mean()), 4) if not mi_values.empty else 0.0
    rating = mi_to_sqale_rating(avg_mi) if not mi_values.empty else "E"
    return pd.DataFrame([
        {"metric_name": "Maintainability_Index", "metric_value": avg_mi},
        {"metric_name": "Maintainability_Rating", "metric_value": rating},
    ])


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW RADON OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

REPO_STATS = compute_repository_stats(REPO_PATH, discover_python_files(REPO_PATH))
logger.info(f'Repository ready at: {REPO_PATH}')
print(f"Repository: {REPO_STATS['repository_name']}")
print(f"Size (Python files): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Directories: {REPO_STATS['directory_count']:,}")
print(f"Python files: {REPO_STATS['python_file_count']:,}")


## Section 5 — Discover Python Files


In [ ]:
PYTHON_FILES = discover_python_files(REPO_PATH)
INVENTORY_CSV = OUTPUT_PATH / 'python_files_inventory.csv'
save_python_inventory(PYTHON_FILES, INVENTORY_CSV)

print(f'Total Python Files Found: {len(PYTHON_FILES)}')
print(f'Saved inventory to: {INVENTORY_CSV}')


## Section 6 — Execute Radon

Run Radon CC, MI, Raw, and Halstead commands. Preserve stdout/stderr exactly as emitted.


In [ ]:
RADON_CONSOLE_CHUNKS: list[str] = []
RADON_JSON: dict[str, str] = {}

if not PYTHON_FILES:
    logger.error('No Python files discovered; skipping Radon execution.')
else:
    for subcommand, extra in [('cc', ['-a']), ('mi', []), ('raw', []), ('hal', [])]:
        text_cmd = build_radon_command(REPO_PATH, subcommand, json_output=False, extra_args=extra)
        json_cmd = build_radon_command(REPO_PATH, subcommand, json_output=True, extra_args=extra)
        text_stdout, text_stderr, text_code = run_radon_command(text_cmd, logger, stream_raw=STREAM_RAW_OUTPUT)
        json_stdout, json_stderr, json_code = run_radon_command(json_cmd, logger, stream_raw=False)
        RADON_CONSOLE_CHUNKS.append(f'===== radon {subcommand} (text) =====\n' + combine_raw_streams(text_stdout, text_stderr))
        RADON_CONSOLE_CHUNKS.append(f'===== radon {subcommand} (json) =====\n' + combine_raw_streams(json_stdout, json_stderr))
        RADON_JSON[subcommand] = json_stdout
        if text_code not in (0, 1):
            logger.error(f'Radon {subcommand} text run exited with code {text_code}', file=subcommand)
        if json_code not in (0, 1):
            logger.error(f'Radon {subcommand} json run exited with code {json_code}', file=subcommand)

logger.info('Radon execution complete.')


## Section 7 — Raw Output Extraction


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'radon_raw_console_output.txt'
CC_JSON_PATH = OUTPUT_PATH / 'radon_cc_output.json'
MI_JSON_PATH = OUTPUT_PATH / 'radon_mi_output.json'
RAW_JSON_PATH = OUTPUT_PATH / 'radon_raw_output.json'
HAL_JSON_PATH = OUTPUT_PATH / 'radon_halstead_output.json'

CONSOLE_PATH.write_text('\n'.join(RADON_CONSOLE_CHUNKS), encoding='utf-8')
CC_JSON_PATH.write_text(RADON_JSON.get('cc', ''), encoding='utf-8')
MI_JSON_PATH.write_text(RADON_JSON.get('mi', ''), encoding='utf-8')
RAW_JSON_PATH.write_text(RADON_JSON.get('raw', ''), encoding='utf-8')
HAL_JSON_PATH.write_text(RADON_JSON.get('hal', ''), encoding='utf-8')

logger.info('Saved Radon raw JSON and console outputs.')
preview_raw_output(CONSOLE_PATH.read_text(encoding='utf-8'), RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 8 — Parse Results


In [ ]:
CC_DF = parse_cc_results(parse_json_payload(RADON_JSON.get('cc', '')))
MI_DF = parse_mi_results(parse_json_payload(RADON_JSON.get('mi', '')))
RAW_DF = parse_raw_results(parse_json_payload(RADON_JSON.get('raw', '')))
HAL_DF = parse_halstead_results(parse_json_payload(RADON_JSON.get('hal', '')))

CC_DF.to_csv(OUTPUT_PATH / 'cyclomatic_complexity_results.csv', index=False)
MI_DF.to_csv(OUTPUT_PATH / 'maintainability_index_results.csv', index=False)
RAW_DF.to_csv(OUTPUT_PATH / 'raw_metrics_results.csv', index=False)
HAL_DF.to_csv(OUTPUT_PATH / 'halstead_metrics_results.csv', index=False)

logger.info(f'Parsed CC rows={len(CC_DF)}, MI rows={len(MI_DF)}, Raw rows={len(RAW_DF)}, Halstead rows={len(HAL_DF)}')


## Section 9 — Maintainability Rating Computation

SQALE A–E mapping from average Maintainability Index.


In [ ]:
SUMMARY_DF = compute_maintainability_summary(MI_DF)
SUMMARY_CSV = OUTPUT_PATH / 'maintainability_rating_summary.csv'
SUMMARY_DF.to_csv(SUMMARY_CSV, index=False)

logger.info(f'Saved maintainability rating summary: {SUMMARY_CSV}')
display(SUMMARY_DF)


## Section 10 — Summary Dashboard


In [ ]:
avg_mi = float(SUMMARY_DF.loc[SUMMARY_DF['metric_name'] == 'Maintainability_Index', 'metric_value'].iloc[0])
rating = str(SUMMARY_DF.loc[SUMMARY_DF['metric_name'] == 'Maintainability_Rating', 'metric_value'].iloc[0])
avg_cc = round(float(pd.to_numeric(CC_DF['complexity'], errors='coerce').dropna().mean()), 4) if not CC_DF.empty else 0.0
total_loc = int(pd.to_numeric(RAW_DF['loc'], errors='coerce').fillna(0).sum()) if not RAW_DF.empty else 0

summary_df = pd.DataFrame([
    {'Metric': 'Total Python Files', 'Value': len(PYTHON_FILES)},
    {'Metric': 'Average Cyclomatic Complexity', 'Value': avg_cc},
    {'Metric': 'Average Maintainability Index', 'Value': avg_mi},
    {'Metric': 'Maintainability Rating', 'Value': rating},
    {'Metric': 'Total LOC', 'Value': total_loc},
])
display(summary_df)

deliverables = [
    CONSOLE_PATH, CC_JSON_PATH, MI_JSON_PATH, RAW_JSON_PATH, HAL_JSON_PATH,
    OUTPUT_PATH / 'cyclomatic_complexity_results.csv',
    OUTPUT_PATH / 'maintainability_index_results.csv',
    OUTPUT_PATH / 'raw_metrics_results.csv',
    OUTPUT_PATH / 'halstead_metrics_results.csv',
    SUMMARY_CSV, INVENTORY_CSV, ERROR_LOG_PATH,
]
print('\nDeliverables:')
for path in deliverables:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {path}")


## Section 11 — Error Handling


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 12 — Deliverables

```text
outputs/
├── radon_cc_output.json
├── radon_mi_output.json
├── radon_raw_output.json
├── radon_halstead_output.json
├── radon_raw_console_output.txt
├── cyclomatic_complexity_results.csv
├── maintainability_index_results.csv
├── raw_metrics_results.csv
├── halstead_metrics_results.csv
├── maintainability_rating_summary.csv
├── python_files_inventory.csv
└── error_log.txt
```
